# 공공데이터 목록 기반 지능형 거버넌스 BR 마이닝
## KDD Process — BI에서 BR로의 전환 (교수님 지침 반영)

**데이터 소스**: 공공데이터활용지원센터_공공데이터포털 목록개방현황_20260531.csv (95,903건)

### 분석 파이프라인
```
Step 1 : 데이터 로딩 & EDA
Step 2 : Entropy 기반 변수 선택 (AOI — 1-D)
Step 3 : 이산화 L0~L4 (수치형 → 범주형)
Step 4 : Roll-up (분류체계 대분류화)
Step 5 : Subspace Focusing (대분류별 분할)
Step 6 : 트랜잭션 DB 변환
Step 7 : FP-growth 빈발 패턴 탐사
Step 8 : BR 추출 (confidence ≥ 0.95) & BI (0.75~0.95)
Step 9 : 우선순위 Ranking
Step 10: PBR 정형화 (IF-THEN 룰셋)
```

### 핵심 원칙 (교수님 지침)
- **N-D 클러스터링 금지** → 1-D 기준 단일 지표만 사용
- **데이터 병합 금지** → 관계 정의 중심
- **수치 묘사 지양** → 이산화 후 범주형 분석
- **신뢰도 ≥ 95%** → BR / **75~95%** → BI

## 환경 설정

In [ ]:
!pip install mlxtend -q

import pandas as pd
import numpy as np
from scipy.stats import entropy as scipy_entropy
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

print('라이브러리 로드 완료')

## Step 1. 데이터 로딩 & EDA

In [ ]:
import os

# ── 경로 설정 ──────────────────────────────────────────
# 방법 A: Google Drive 마운트
DRIVE_PATH = '/content/drive/MyDrive/공공데이터활용지원센터_공공데이터포털 목록개방현황_20260531.csv'

# 방법 B: 직접 업로드 (Drive 없을 때)
if not os.path.exists(DRIVE_PATH):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except:
        pass

if not os.path.exists(DRIVE_PATH):
    from google.colab import files
    print('CSV 파일을 업로드하세요')
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]
else:
    CSV_PATH = DRIVE_PATH

df_raw = pd.read_csv(CSV_PATH, encoding='utf-8-sig', low_memory=False)
print(f'로드 완료: {df_raw.shape[0]:,}행 × {df_raw.shape[1]}열')
df_raw.head(3)

In [ ]:
print('=== 컬럼 목록 ===')
for c in df_raw.columns:
    nun = df_raw[c].nunique()
    null_r = df_raw[c].isnull().mean() * 100
    print(f'  {c:<30} unique={nun:>6,}  null={null_r:.1f}%')

print('\n=== 목록유형 분포 ===')
print(df_raw['목록유형'].value_counts())

print('\n=== 대분류 분포 (상위 15) ===')
df_raw['대분류'] = df_raw['분류체계'].str.split(' - ').str[0].str.strip()
print(df_raw['대분류'].value_counts().head(15))

## Step 2. Entropy 기반 변수 선택 (AOI — 1-D)
> 교수님 지침: 단일 지표(1-D)로만 컬럼 평가. N-D 복합 지표 금지.

In [ ]:
# 분석 대상 컬럼 (범주형)
CAT_COLS = [
    '목록유형',
    '대분류',
    '업데이트 주기',
    '확장자(데이터포맷)',
    '국가중점여부',
    '표준데이터여부',
    '비용부과유무',
    '제공형태',
]

# 수치형 컬럼 (이산화 대상)
NUM_COLS = ['다운로드_활용신청건수', '조회수']

# 1-D Entropy 계산 (각 컬럼 독립적으로)
entropy_results = {}
for col in CAT_COLS:
    vc = df_raw[col].dropna().value_counts(normalize=True)
    e = scipy_entropy(vc, base=2)
    entropy_results[col] = round(e, 4)

# 수치형 → 5구간 임시 이산화 후 엔트로피
for col in NUM_COLS:
    vals = pd.to_numeric(df_raw[col], errors='coerce').dropna()
    if len(vals) > 0:
        bins = pd.qcut(vals, q=5, duplicates='drop')
        vc = bins.value_counts(normalize=True)
        e = scipy_entropy(vc, base=2)
        entropy_results[col] = round(e, 4)

ent_df = pd.Series(entropy_results).sort_values(ascending=False).to_frame('Entropy')
ent_df['선택'] = ent_df['Entropy'].apply(lambda x: '✅ 선택' if x >= 1.0 else ('⚠️ 보조' if x >= 0.3 else '❌ 제거'))

print('=== 1-D Entropy 기반 변수 선택 결과 ===')
print(ent_df.to_string())

# 선택된 컬럼만 유지
SELECTED_COLS = [c for c in ent_df[ent_df['선택'] == '✅ 선택'].index]
print(f'\n선택된 컬럼: {SELECTED_COLS}')

## Step 3. 이산화 (Discretization) L0 ~ L4
> 교수님 지침: 수치 데이터는 이산화하여 범주형으로 변환. AOI 적용.

In [ ]:
df = df_raw.copy()

LEVEL_LABELS = ['L0(최소)', 'L1(하)', 'L2(중)', 'L3(상)', 'L4(최대)']

for col in NUM_COLS:
    vals = pd.to_numeric(df[col], errors='coerce')
    col_disc = col + '_레벨'
    try:
        df[col_disc] = pd.qcut(vals, q=5, labels=LEVEL_LABELS, duplicates='drop')
    except ValueError:
        # 중복값이 많으면 cut으로 대체
        df[col_disc] = pd.cut(vals, bins=5, labels=LEVEL_LABELS)
    df[col_disc] = df[col_disc].astype(str).replace('nan', '-')

    print(f'[{col}] 이산화 결과:')
    print(df[col_disc].value_counts().to_string())
    print()

# 수치형 원본 제거 (이산화 값만 사용)
DISC_COLS = [c + '_레벨' for c in NUM_COLS]
print('이산화 완료. 수치형 컬럼 → 레벨 컬럼으로 대체')

## Step 4. Roll-up — 개념 계층화
> 교수님 지침: 상위 개념(대분류)으로 롤업하여 패턴 명확화.

In [ ]:
# 분류체계: 'A - B' → 대분류 A만 사용 (Roll-up)
df['분류_대분류'] = df['분류체계'].str.split(' - ').str[0].str.strip()

# 업데이트 주기 정리
df['업데이트주기_정'] = df['업데이트 주기'].replace({'-': '미정'})

# 확장자 정규화 (대소문자, 복합형 단순화)
def normalize_ext(x):
    if pd.isna(x) or x == '-': return '기타'
    x = str(x).lower().strip()
    if 'json' in x and 'xml' in x: return 'JSON+XML'
    if x in ['csv']: return 'CSV'
    if x in ['xlsx', 'xls']: return 'Excel'
    if x in ['pdf']: return 'PDF'
    if x in ['json']: return 'JSON'
    if x in ['xml']: return 'XML'
    if x in ['hwp', 'hwpx']: return 'HWP'
    if x in ['jpg', 'png', 'gif', 'jpeg']: return '이미지'
    return '기타'

df['확장자_정'] = df['확장자(데이터포맷)'].apply(normalize_ext)

print('대분류 Roll-up 결과:')
print(df['분류_대분류'].value_counts().head(15))

print('\n확장자 정규화 결과:')
print(df['확장자_정'].value_counts())

## Step 5. Subspace Focusing
> 교수님 지침: 전체가 아닌 특정 부분공간(Subspace) 집중 분석으로 신뢰도 확보.

In [ ]:
# 대분류별 데이터 수
subspace_counts = df['분류_대분류'].value_counts()
print('=== 서브스페이스 목록 (대분류) ===')
print(subspace_counts.to_string())

# 분석할 서브스페이스 선택 (상위 5개 + 전체)
TOP_SUBSPACES = subspace_counts.head(5).index.tolist()
print(f'\n분석 대상 서브스페이스: {TOP_SUBSPACES}')

# 서브스페이스 딕셔너리
subspaces = {'전체': df}
for ss in TOP_SUBSPACES:
    subspaces[ss] = df[df['분류_대분류'] == ss].copy()
    print(f'  [{ss}] {len(subspaces[ss]):,}건')

## Step 6. 트랜잭션 DB 변환

In [ ]:
# 트랜잭션 아이템으로 사용할 컬럼
# 교수님 지침: 1-D 기준 단일 속성만 아이템화
ITEM_COLS = [
    '목록유형',
    '업데이트주기_정',
    '확장자_정',
    '국가중점여부',
    '표준데이터여부',
    '비용부과유무',
    '다운로드_활용신청건수_레벨',
    '조회수_레벨',
]

def make_transactions(df_sub, item_cols, subspace_name):
    """데이터프레임 → 트랜잭션 리스트 변환"""
    transactions = []
    for _, row in df_sub.iterrows():
        items = []
        for col in item_cols:
            val = row.get(col, None)
            if pd.notna(val) and str(val) not in ['-', 'nan', '미정', '']:
                items.append(f'{col}={val}')
        if items:
            transactions.append(items)
    return transactions

# 전체 트랜잭션 변환 미리보기
sample_tx = make_transactions(df.head(5), ITEM_COLS, '전체')
print('=== 트랜잭션 샘플 (5건) ===')
for i, tx in enumerate(sample_tx):
    print(f'  T{i+1}: {tx}')

## Step 7. FP-growth 빈발 패턴 탐사
> 교수님 지침: FP-tree 알고리즘 사용. 다차원 동시 출현 패턴 포착.

In [ ]:
def run_fpgrowth(df_sub, item_cols, subspace_name, min_support=0.1):
    """FP-growth 실행 → 빈발 패턴 반환"""
    print(f'\n[{subspace_name}] FP-growth 실행 ({len(df_sub):,}건, minsup={min_support})')

    transactions = make_transactions(df_sub, item_cols, subspace_name)
    if len(transactions) == 0:
        print('  트랜잭션 없음')
        return pd.DataFrame(), pd.DataFrame()

    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    df_te = pd.DataFrame(te_array, columns=te.columns_)

    freq_items = fpgrowth(df_te, min_support=min_support, use_colnames=True)
    freq_items['itemsets_str'] = freq_items['itemsets'].apply(lambda x: ', '.join(sorted(x)))
    freq_items = freq_items.sort_values('support', ascending=False)

    print(f'  빈발 패턴 {len(freq_items)}개 발견')
    print(freq_items.head(10)[['support', 'itemsets_str']].to_string())

    return freq_items, df_te

# 서브스페이스별 FP-growth 실행
fp_results = {}
te_results = {}

for ss_name, ss_df in subspaces.items():
    fp, te_df = run_fpgrowth(ss_df, ITEM_COLS, ss_name, min_support=0.1)
    fp_results[ss_name] = fp
    te_results[ss_name] = te_df

## Step 8. BR 추출 — 신뢰도 기반 BI / BR 분류
> - **BR (Business Rule)**: confidence ≥ 0.95
> - **BI (Business Intelligence)**: 0.75 ≤ confidence < 0.95

In [ ]:
def extract_rules(freq_items, te_df, ss_name, min_confidence=0.75, min_lift=1.0):
    """연관 규칙 추출 → BI/BR 분류"""
    if len(freq_items) == 0 or len(te_df) == 0:
        return pd.DataFrame()

    rules = association_rules(freq_items, metric='confidence',
                               min_threshold=min_confidence)
    rules = rules[rules['lift'] >= min_lift].copy()

    rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ' AND '.join(sorted(x)))
    rules['consequents_str'] = rules['consequents'].apply(lambda x: ' AND '.join(sorted(x)))
    rules['subspace'] = ss_name

    # BI vs BR 분류
    rules['분류'] = rules['confidence'].apply(
        lambda c: 'BR' if c >= 0.95 else 'BI'
    )

    # IF-THEN 형식 생성
    rules['IF_THEN'] = rules.apply(
        lambda r: f"IF [{r['antecedents_str']}] THEN [{r['consequents_str']}]", axis=1
    )

    br_count = (rules['분류'] == 'BR').sum()
    bi_count = (rules['분류'] == 'BI').sum()
    print(f'  [{ss_name}] 규칙 추출: BR={br_count}개, BI={bi_count}개')
    return rules

all_rules = []
for ss_name in subspaces.keys():
    if ss_name in fp_results and len(fp_results[ss_name]) > 0:
        rules = extract_rules(fp_results[ss_name], te_results[ss_name], ss_name)
        if len(rules) > 0:
            all_rules.append(rules)

if all_rules:
    rules_df = pd.concat(all_rules, ignore_index=True)
    print(f'\n총 규칙: {len(rules_df)}개 (BR: {(rules_df["분류"]=="BR").sum()}, BI: {(rules_df["분류"]=="BI").sum()})')
else:
    print('추출된 규칙 없음')
    rules_df = pd.DataFrame()

## Step 9. 우선순위 Ranking
> 교수님 지침: 분석 결과를 나열만 하지 말고, 수치 지표로 우선순위 자동 산출.

In [ ]:
if len(rules_df) > 0:
    # 우선순위 점수 = support × confidence × lift (정규화)
    rules_df['priority_score'] = (
        rules_df['support'] * rules_df['confidence'] * rules_df['lift']
    )
    rules_df['priority_score'] = (
        (rules_df['priority_score'] - rules_df['priority_score'].min()) /
        (rules_df['priority_score'].max() - rules_df['priority_score'].min() + 1e-10)
    )
    rules_df['rank'] = rules_df['priority_score'].rank(ascending=False).astype(int)
    rules_df = rules_df.sort_values('rank')

    print('=== 우선순위 TOP 20 규칙 ===')
    display_cols = ['rank', '분류', 'subspace', 'support', 'confidence', 'lift', 'priority_score', 'IF_THEN']
    print(rules_df[display_cols].head(20).to_string(index=False))
else:
    print('규칙 없음')

## Step 10. PBR 정형화 — IF-THEN 룰셋
> 교수님 지침: 도출된 규칙을 IF-THEN 구조로 정형화하고 신뢰도(%) 제시.

In [ ]:
if len(rules_df) > 0:
    br_rules = rules_df[rules_df['분류'] == 'BR'].sort_values('rank')
    bi_rules = rules_df[rules_df['분류'] == 'BI'].sort_values('rank')

    print('=' * 80)
    print('[ BR (Business Rules) — 자동 거버넌스 통제 룰 ]')
    print('=' * 80)
    for _, r in br_rules.head(20).iterrows():
        print(f'  [Rank {r["rank"]:>3}] [서브스페이스: {r["subspace"]}]')
        print(f'  {r["IF_THEN"]}')
        print(f'  → confidence={r["confidence"]:.1%}, support={r["support"]:.1%}, lift={r["lift"]:.2f}')
        print()

    print('=' * 80)
    print('[ BI (Business Intelligence) — 업무 지식 ]')
    print('=' * 80)
    for _, r in bi_rules.head(10).iterrows():
        print(f'  [Rank {r["rank"]:>3}] [서브스페이스: {r["subspace"]}]')
        print(f'  {r["IF_THEN"]}')
        print(f'  → confidence={r["confidence"]:.1%}, support={r["support"]:.1%}, lift={r["lift"]:.2f}')
        print()

    # CSV 저장
    out_path = '/content/drive/MyDrive/BR_Mining_Results.csv'
    if not os.path.exists('/content/drive/MyDrive'):
        out_path = 'BR_Mining_Results.csv'
    rules_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'\n결과 저장: {out_path}')
else:
    print('추출된 규칙 없음 — min_support 또는 min_confidence 조정 필요')

## 부록. Subspace별 요약 대시보드

In [ ]:
if len(rules_df) > 0:
    summary = rules_df.groupby(['subspace', '분류']).agg(
        규칙수=('rank', 'count'),
        평균신뢰도=('confidence', 'mean'),
        최대신뢰도=('confidence', 'max'),
        평균support=('support', 'mean'),
    ).round(3)
    print('=== 서브스페이스별 BR/BI 요약 ===')
    print(summary.to_string())

    print('\n=== BR 상위 패턴 요약 ===')
    br_top = rules_df[rules_df['분류']=='BR'].groupby('subspace').head(1)[
        ['subspace','IF_THEN','confidence','support']
    ]
    for _, r in br_top.iterrows():
        print(f'  [{r["subspace"]}] {r["IF_THEN"]}')
        print(f'    → conf={r["confidence"]:.1%}, supp={r["support"]:.1%}')
        print()